In [1]:
# import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import random
# import networkx as nx

from plotly.offline import init_notebook_mode, iplot
from plotly.graph_objs import *
from plotly.subplots import make_subplots

from fivetwo.deck import Deck
from fivetwo.shuffler import Shuffler
from fivetwo.stats import Statistics

### Creating a Deck

In [2]:
import platform

if platform.node() == "DESKTOP-1H9GDS2":
    deck = np.array([random.randint(1, 8) for _ in range(99)])
else:
    import sys
    sys.path.insert(0, r"C:\Users\colli\OneDrive\Documents\Code\git\card_converter\src")

    from deck_conversion import Converter

    premade_deck = Converter(file_name = r"C:\Users\colli\OneDrive\Documents\Code\git\card_converter\decklists\makin'_copies.txt")
    premade_deck = premade_deck.converted
    deck = Deck(premade_deck)

index_list = np.arange(0, len(deck), dtype=int)

### Operator Method

#### Operator Generation from Any Shuffled List

In [3]:
def generate_operator(initial_state: list, shuffled_state: list):
    operator = np.zeros((len(initial_state), len(initial_state)))
    operator[index_list, shuffled_state] = 1

    return operator

In [4]:
class CantBeShuffledError:
    pass

In [5]:
def check_cards_and_piles(number_of_cards: int, number_of_piles: int):
    """
    Checks to see if the number of cards and the number of piles are valid for the shuffle methods.
    ---
    Parameters:
        number_of_cards, int: How large the deck is.
        number_of_piles, int: How many piles the cards will be distributed among.
    """

    if number_of_cards <= 1:
        raise CantBeShuffledError
    if number_of_piles < 1:
        raise CantBeShuffledError

#### Pile Shuffle

In [6]:
def pile_shuffle_indices(
        number_of_cards: int,
        number_of_piles: int,
        random_pile_placement: bool = False,
        random_pile_pickup: bool = False):
    """
    Distributes the decks into piles and then recombines the piles to form the shuffled deck.
    ---
    Parameters:
        number_of_cards, int: The size of the deck being shuffled.
        number_of_piles, int: The number of piles to sort the deck out into.
        rand, bool: Determines if the shuffle is random or not.
    Return:
        shuffled_indices, np.array: The indices of the new state of the deck. Apply to an array of the actual deck to
                                    get the shuffled result.
    """

    check_result = check_cards_and_piles(number_of_cards, number_of_piles)
    if check_result is not None:
        return check_result

    card_indices = np.arange(0, len(deck)) + 1
    piles = np.zeros(shape=(number_of_piles, number_of_cards))
    pile_indices = np.zeros(number_of_piles, dtype=int)

    pile_num = 0
    for _ in range(number_of_cards):
        # Adding a card to a pile.
        pile = pile_num
        if random_pile_placement:
            pile_num = random.randint(0,number_of_piles-1)
        piles[pile, pile_indices[pile]] = card_indices[0]
        pile_indices[pile] += 1
        
        # Prevents a card from being placed in two piles simultaneously.
        card_indices = np.delete(card_indices, 0)
        
        # Moving to the next pile.
        pile_num +=1
        if pile_num == number_of_piles:
            pile_num = 0
    
    if random_pile_pickup:
        rng = np.random.default_rng()
        piles = rng.permutation(piles, axis=0)
    piles = np.flip(piles, axis=1)

    shuffled_indices = piles.flatten()
    shuffled_indices = np.delete(shuffled_indices, np.where(shuffled_indices == 0))
    shuffled_indices -= 1

    return np.array(shuffled_indices, dtype=int)

In [ ]:
def user_driven_pile_shuffle():
    """
    Asks the user for each step of shuffling the deck in a pile-shuffle method.
    """

    def guarantee_input(prompt: str, desired_type: type):
        """
        Guarantees that the input provided by the user can be converted to the right type.
        Does so through persistent while loops.
        """

        user_input = None
        while type(user_input) is not desired_type:
            user_input = input(prompt + " ") # <ws> string added to make output look good.
            try:
                user_input = desired_type(user_input)
            except NotImplementedError:
                print("Can't convert to this type. Make sure it's callable.")
            except ValueError:
                pass

    size_of_deck = guarantee_input("What is the size of your deck?", int)

### Investigating the Transformations of States

#### State Similarites After Transformation - TODO: The Dot Products don't Make Sense; Investigate

In [10]:
angles = []
iterations = 10000

initial_state = deck
operator = generate_operator(index_list, pile_shuffle_indices(len(deck), 11, True, True))
for _ in range(iterations):
    print(f"IS: {initial_state}")
    new_state = np.matmul(operator, initial_state)
    print(f"NS: {new_state}")
    dot = np.matmul(initial_state, new_state)
    print(f"Dot Product: {dot}")
    angles.append(np.acos(np.matmul(initial_state, new_state) / sum(initial_state ** 2)))
    initial_state = new_state

fig = px.polar(angles)
fig.show()

IS: [4 3 2 7 3 7 4 8 6 7 2 4 2 6 2 2 5 8 7 6 1 7 7 6 5 2 4 8 1 7 6 6 8 3 6 3 5
 8 6 3 6 7 1 2 2 2 8 2 8 3 2 5 4 4 7 6 8 4 4 2 4 4 5 1 5 6 8 6 6 7 4 1 1 7
 7 7 3 2 6 7 3 6 8 5 3 4 8 5 5 5 8 8 2 8 2 8 4 5 3]
NS: [2. 8. 6. 7. 5. 7. 6. 6. 4. 4. 2. 1. 1. 5. 8. 7. 4. 1. 2. 8. 3. 7. 7. 7.
 3. 4. 8. 8. 8. 2. 8. 3. 7. 2. 6. 5. 8. 2. 1. 2. 8. 6. 8. 2. 2. 5. 4. 2.
 3. 6. 5. 4. 7. 3. 5. 6. 3. 7. 6. 1. 2. 6. 5. 4. 2. 2. 4. 5. 4. 4. 4. 4.
 7. 5. 7. 6. 6. 2. 4. 3. 6. 7. 5. 6. 7. 8. 8. 5. 8. 1. 8. 3. 3. 8. 2. 6.
 6. 7. 3.]
Dot Product: 2397.0
IS: [2. 8. 6. 7. 5. 7. 6. 6. 4. 4. 2. 1. 1. 5. 8. 7. 4. 1. 2. 8. 3. 7. 7. 7.
 3. 4. 8. 8. 8. 2. 8. 3. 7. 2. 6. 5. 8. 2. 1. 2. 8. 6. 8. 2. 2. 5. 4. 2.
 3. 6. 5. 4. 7. 3. 5. 6. 3. 7. 6. 1. 2. 6. 5. 4. 2. 2. 4. 5. 4. 4. 4. 4.
 7. 5. 7. 6. 6. 2. 4. 3. 6. 7. 5. 6. 7. 8. 8. 5. 8. 1. 8. 3. 3. 8. 2. 6.
 6. 7. 3.]
NS: [2. 8. 7. 7. 4. 7. 4. 5. 3. 7. 5. 8. 3. 7. 8. 3. 6. 8. 2. 3. 6. 4. 7. 2.
 5. 6. 3. 3. 4. 2. 2. 5. 2. 4. 8. 6. 5. 2. 4. 2. 7. 3. 6. 2. 3. 5. 7. 5.
 2. 8. 3. 

AttributeError: module 'plotly.express' has no attribute 'polar'

#### Visualizing Connections - TODO Finish changing matplotlib to plotly for graphing

In [ ]:
new_state = np.array(np.matmul(operator, initial_state), dtype=int)
node_connections = list(zip(new_state, np.roll(new_state, -1)))[:-1] # Removing the last because the pairings are not cyclic.
directed_graph = nx.MultiDiGraph(node_connections)
nodes = sorted(directed_graph.nodes())
base_theta = 2 * np.pi / len(nodes)
positions = {node: (np.cos(i * base_theta), np.sin(i * base_theta)) for (i, node) in enumerate(nodes)}

plt.figure(figsize=(8,8))
nx.draw_networkx_nodes(directed_graph, positions)
nx.draw_networkx_edges(directed_graph, positions)
nx.draw_networkx_labels(directed_graph, positions)

NameError: name 'nx' is not defined

#### Visualizing Card Transitions (a rehashing of the plot above) - TODO Finish changing matplotlib to plotly for graphing

In [ ]:
out_edges = np.zeros((len(nodes), len(nodes)))

for edge in directed_graph.out_edges():
    out_edges[edge[0] - 1][edge[1] - 1] += 1

f, axs = plt.subplots()
plot = axs.imshow(out_edges, aspect='auto')
# axs.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False,
#                labelbottom=False, labeltop=False, labelleft=False, labelright=False)
plt.colorbar(plot)
plt.title("Out Edges (R2C), In Edges (C2R)")

### Visualizing Card Neighborhoods

In [ ]:
iterations = 10000

node_data = np.zeros((iterations, len(nodes), len(nodes)))

state = initial_state
for iteration in range(iterations):
    state = np.array(np.matmul(operator, state), dtype=int)
    node_connections = list(zip(state, np.roll(state, -1)))[:-1] # Removing the last because the pairings are not cyclic.
    directed_graph = nx.MultiDiGraph(node_connections)
    out_edges = np.zeros((len(nodes), len(nodes)))
    for edge in directed_graph.out_edges():
        out_edges[edge[0] - 1][edge[1] - 1] += 1
    node_data[iteration] = out_edges

In [ ]:
f, axs = plt.subplots(8, 8, figsize=(16,9))
for i in range(axs.shape[0]):
    for j in range(axs.shape[1]):
        axs[i,j].hist(node_data[:,i,j], align="left")
        axs[i,j].set_title(f"Transitions from {i+1} to {j+1}")

f.suptitle("Card Neighborhoods")
plt.tight_layout()
plt.show()

## Calculating Amortizations

In [ ]:
remaining_loans = np.array([4857.79, 5441.15, 2999.05, 1580.28, 4711.94])
interest_rates = np.array([6.550, 6.550, 2.400, 5.800, 2.860]) / 100
days_to_pay_off_with_monthly_payments = np.array([56, 57, 55, 56, 57])
interest_payments_over_years = remaining_loans * interest_rates * (days_to_pay_off_with_monthly_payments * 30) / 365
monthly_payments = [396.88]  

In [ ]:
print(f"Remaining loan amounts: ${', $'.join(map(str, remaining_loans))}")
print(f"Total remaining principal payment amount: ${sum(remaining_loans)}")
print(f"Interest per loan per day I'll pay if I keep these loans: ${', $'.join(map(str, interest_payments_over_years))}")
print(f"Total interest payment amount: ${sum(interest_payments_over_years)}")
print(f"Combined monthly payment amount with current loans: ${sum(monthly_payments)}")